# YOLOv8m + MLCA (Colony-YOLO) sobre el subdataset de AGAR

Prueba: **YOLOv8m pre-entrenado en COCO** con los **hiperparámetros** de la versión anterior, agregando el mecanismo de atención del paper Colony-YOLO: **C2f-MLCA en el neck** (MLCA dentro de cada Bottleneck, Figura 6). Ahora se entrena y evalúa sobre el **subdataset del AGAR oficial** (solo *E. coli*, *P. aeruginosa* y *S. aureus*).

Se entrenan dos variantes con la misma configuración, para que la única diferencia sea el MLCA:

| variante | qué es |
|---|---|
| `base` | YOLOv8m tal cual (control) |
| `mlca` | YOLOv8m con los 4 C2f del neck → C2f-MLCA (8 MLCA). El backbone no cambia |

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno → GPU T4`. Después ejecutá las celdas en orden.

**Tiempo:** con AGAR (5823 fotos de train) cada corrida tarda **~3 h** en una T4 (≈ 4 min por época). Con `SEEDS = [0]` son ~6 h para las dos variantes.

**Qué se guarda y cuándo (nuevo):**
* Mientras entrena, cada `SYNC_EVERY` épocas se copia a Drive la carpeta de la corrida (`last.pt`, `best.pt`, `results.csv`, gráficas).
* Si Colab se desconecta, volvé a ejecutar las secciones **1 a 8 en orden**: la corrida interrumpida **se retoma desde la última época guardada** y las terminadas se saltean.
* La sección 10 muestra las gráficas (también de corridas a medias) y la sección 11 baja a tu computadora un `.zip` por corrida con `best.pt` y todas las gráficas.

## 1 · GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2 · Librerías
Misma versión de ultralytics que usó Belén (8.4.163).

In [ ]:
!pip install -q ultralytics==8.4.163 roboflow
import ultralytics
print('ultralytics', ultralytics.__version__)

## 3 · Configuración

La API key de Roboflow **no** se escribe en el notebook. La celda la busca en los *Secrets* de Colab (ícono de llave 🔑 a la izquierda, nombre `ROBOFLOW_API_KEY`) y, si no está, la pide por teclado.

In [ ]:
SEEDS     = [0]                # una corrida por seed y por variante (agregá 1, 2 si hay tiempo: ~3 h cada una)
VARIANTES = ['base', 'mlca']
BATCH     = 32                 # entra en los 15 GB de la T4
EPOCHS    = 100
DRIVE_DIR = '/content/drive/MyDrive/yolov8m_mlca'

SYNC_EVERY = 2                 # cada cuántas épocas se copia la corrida a Drive (1 = cada época)
VAL_SUBSET = None              # None = validar con todo valid (1344 fotos, ~40 s por época).
                               # Un número (ej. 400) valida cada época con ese subconjunto fijo: ahorra ~25 min por corrida.
                               # La evaluación final (sección 9) siempre usa valid y test completos.

# Hiperparámetros de la versión anterior (best_v8.pt)
HER_ARGS = dict(patience=15, imgsz=640, optimizer='Adam', lr0=0.0001, lrf=0.01, momentum=0.937,
                weight_decay=0.0001, warmup_epochs=3.0, warmup_momentum=0.8, warmup_bias_lr=0.1,
                box=7.5, cls=0.5, dfl=1.5, nbs=64, hsv_h=0.0, hsv_s=0.0, hsv_v=0.15, degrees=12.0,
                translate=0.1, scale=0.1, shear=0.0, perspective=0.0, flipud=0.0, fliplr=0.5,
                mosaic=1.0, mixup=0.0, copy_paste=0.0, close_mosaic=10, erasing=0.4, cos_lr=False,
                deterministic=True, amp=True, iou=0.7, max_det=300, pretrained=True, workers=8)

## 4 · Google Drive

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.makedirs(DRIVE_DIR, exist_ok=True)
print('resultados en', DRIVE_DIR)

## 5 · Dataset: subdataset de AGAR (3 especies)

Se descarga de Roboflow (`agar_subdataset`, versión 1). **Las clases se leen del `data.yaml` descargado** (no se escriben a mano), para que los nombres coincidan siempre con los ids de las etiquetas.

Los nombres de archivo de AGAR son numéricos y **no contienen la especie**, así que la especie verdadera de cada placa se obtiene de sus etiquetas (clase mayoritaria de las cajas anotadas).

In [ ]:
import os, getpass, shutil, random, yaml
from roboflow import Roboflow

try:
    from google.colab import userdata
    API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    API_KEY = None
if not API_KEY:
    API_KEY = getpass.getpass('API key de Roboflow: ')

ROOT = '/content/agar_subdataset'
rf = Roboflow(api_key=API_KEY.strip())
ds = (rf.workspace('ucuestudiantesproyecto-gmail-com')
        .project('agar_subdataset').version(1).download('yolov8', location=ROOT))
del API_KEY

# clases tal como vienen en el dataset
_names = yaml.safe_load(open(f'{ROOT}/data.yaml'))['names']
NAMES = list(_names.values()) if isinstance(_names, dict) else list(_names)
assert len(NAMES) == 3, NAMES
print('clases:', NAMES)

def write_yaml(path, val):
    with open(path, 'w') as f:
        yaml.safe_dump({'path': ROOT, 'train': 'train/images', 'val': val, 'test': 'test/images',
                        'nc': len(NAMES), 'names': NAMES}, f, sort_keys=False)

DATA_YAML = '/content/data_agar.yaml'                 # valid completo: evaluación final
write_yaml(DATA_YAML, 'valid/images')

TRAIN_YAML = DATA_YAML                                # yaml que usa el entrenamiento
if VAL_SUBSET:
    sub = '/content/agar_val_sub'
    shutil.rmtree(sub, ignore_errors=True)
    os.makedirs(f'{sub}/images'); os.makedirs(f'{sub}/labels')
    fs = sorted(os.listdir(f'{ROOT}/valid/images'))
    random.Random(0).shuffle(fs)
    for f in fs[:VAL_SUBSET]:
        shutil.copy(f'{ROOT}/valid/images/{f}', f'{sub}/images/{f}')
        shutil.copy(f'{ROOT}/valid/labels/{os.path.splitext(f)[0]}.txt', f'{sub}/labels/{os.path.splitext(f)[0]}.txt')
    TRAIN_YAML = '/content/data_agar_train.yaml'
    write_yaml(TRAIN_YAML, f'{sub}/images')
    print(f'validación durante el entrenamiento: {VAL_SUBSET} fotos de valid')

for s in ('train', 'valid', 'test'):
    print(s, len(os.listdir(f'{ROOT}/{s}/images')), 'fotos')   # esperado: train 5823 / valid 1344 / test 1792

## 6 · C2f-MLCA (atención del paper)

* **MLCA** (Mixed Local Channel Attention): atención de canales local (pooling 5×5) + global, cada una con un Conv1d de kernel *k* según la Ec. (3) del paper (g = b = 2).
* **Bottleneck_MLCA**: `x → Conv → Conv → MLCA (→ + x si hay shortcut)`. La atención modula la rama residual (Figura 6).
* **C2f_MLCA**: un C2f cuyos Bottleneck son Bottleneck_MLCA.

ultralytics solo calcula canales y repeticiones para los módulos de una lista interna. Durante la construcción del modelo, el nombre `C2fPSA` (que está en esa lista y YOLOv8m no usa) apunta a `C2f_MLCA`. Así solo llevan atención las capas que el YAML declara como `C2f_MLCA`, y el backbone queda intacto.

**Si se reinicia el entorno, ejecutá esta celda antes de cargar un modelo con MLCA.**

In [ ]:
import math, inspect
import torch
import torch.nn as nn
import torch.nn.functional as F
import ultralytics.nn.tasks as tasks
import ultralytics.nn.modules.block as block
from ultralytics.nn.modules.block import C2f, Bottleneck


class MLCA(nn.Module):
    def __init__(self, channel, local_size=5, gamma=2, b=2, local_weight=0.5):
        super().__init__()
        self.local_size = int(local_size)
        self.local_weight = float(local_weight)
        c = int(channel)
        if c % 2 == 0:                               # Ec. (3): si C es par, C+1
            c += 1
        t = int(abs(math.log2(c) / gamma + b / gamma))
        k = max(t if t % 2 else t + 1, 3)
        self.conv = nn.Conv1d(1, 1, kernel_size=k, padding=(k - 1) // 2, bias=False)
        self.conv_local = nn.Conv1d(1, 1, kernel_size=k, padding=(k - 1) // 2, bias=False)
        self.local_pool = nn.AdaptiveAvgPool2d(self.local_size)
        self.global_pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        b, c, h, w = x.shape
        ls = self.local_size
        local = self.local_pool(x)
        glob = self.global_pool(local)
        t_local = local.view(b, c, -1).transpose(-1, -2).reshape(b, 1, -1)
        t_glob = glob.view(b, c, -1).transpose(-1, -2)
        y_local = self.conv_local(t_local)
        y_glob = self.conv(t_glob)
        att_local = y_local.reshape(b, ls * ls, c).transpose(-1, -2).reshape(b, c, ls, ls).sigmoid()
        att_glob = y_glob.transpose(-1, -2).reshape(b, c, 1, 1).sigmoid().expand(-1, -1, ls, ls)
        att = att_glob * (1 - self.local_weight) + att_local * self.local_weight
        return x * F.interpolate(att, size=(h, w), mode='nearest')


class Bottleneck_MLCA(Bottleneck):
    def __init__(self, c1, c2, shortcut=True, g=1, k=(3, 3), e=0.5):
        super().__init__(c1, c2, shortcut, g, k, e)
        self.mlca = MLCA(c2)

    def forward(self, x):
        y = self.mlca(self.cv2(self.cv1(x)))
        return x + y if self.add else y


class C2f_MLCA(C2f):
    def __init__(self, c1, c2, n=1, shortcut=False, g=1, e=0.5):
        super().__init__(c1, c2, n, shortcut, g, e)
        self.m = nn.ModuleList(
            Bottleneck_MLCA(self.c, self.c, shortcut, g, k=((3, 3), (3, 3)), e=1.0) for _ in range(n))


# registrar (mismo nombre de módulo que en la máquina local: los .pt se pueden cargar en los dos lados)
for cls in (MLCA, Bottleneck_MLCA, C2f_MLCA):
    cls.__module__ = 'ultralytics.nn.modules.block'
    cls.__qualname__ = cls.__name__
    setattr(block, cls.__name__, cls)
    setattr(tasks, cls.__name__, cls)

assert 'C2fPSA' in inspect.getsource(tasks.parse_model), 'esta version de ultralytics cambio parse_model'
if not getattr(tasks.parse_model, '_mlca_v8', False):
    _original_parse_model = tasks.parse_model
    def parse_model(d, ch, verbose=True):
        saved = tasks.C2fPSA
        tasks.C2fPSA = C2f_MLCA
        try:
            return _original_parse_model(d, ch, verbose)
        finally:
            tasks.C2fPSA = saved
    parse_model._mlca_v8 = True
    tasks.parse_model = parse_model
print('C2f-MLCA registrado')

## 7 · Arquitectura y prueba

In [ ]:
from collections import Counter
from ultralytics import YOLO

src = os.path.join(os.path.dirname(ultralytics.__file__), 'cfg', 'models', 'v8', 'yolov8.yaml')
out, in_head, n = [], False, 0
for ln in open(src).read().splitlines():
    if ln.startswith('head:'):
        in_head = True
    if in_head and ', C2f, [' in ln:
        ln, n = ln.replace(', C2f, [', ', C2f_MLCA, ['), n + 1
    if ln.startswith('nc:'):
        ln = 'nc: 3'
    out.append(ln)
assert n == 4
MLCA_YAML = '/content/yolov8m-mlca-agar.yaml'      # el nombre empieza con 'yolov8m' -> escala m
open(MLCA_YAML, 'w').write('\n'.join(out) + '\n')


def make_model(variant):
    if variant == 'base':
        return YOLO('yolov8m.pt')               # exactamente como Belén
    m = YOLO(MLCA_YAML)
    m.load('yolov8m.pt')                       # COCO en todo menos el MLCA
    return m


for v in VARIANTES:
    m = make_model(v)
    k = Counter(type(x).__name__ for x in m.model.modules())
    bb = Counter(type(x).__name__ for x in m.model.model[:10].modules())
    print(f"{v:5s} C2f={k['C2f']} C2f_MLCA={k['C2f_MLCA']} MLCA={k['MLCA']} | MLCA en backbone={bb['MLCA']}")
    if v == 'mlca':
        assert k['C2f_MLCA'] == 4 and k['MLCA'] == 8 and bb['MLCA'] == 0
# esperado:  base C2f=8 C2f_MLCA=0 MLCA=0  |  mlca C2f=4 C2f_MLCA=4 MLCA=8, 0 en el backbone

## 8 · Entrenar (con guardado en Drive y reanudación)

Una corrida por variante y seed, intercaladas (`base_s0`, `mlca_s0`, `base_s1`, ...).

**Cómo se guarda:** cada corrida entrena en el disco local de Colab (`/content/runs_agar`), porque entrenar directo en Drive falla (`OSError: [Errno 95]`). Para no perder nada, un *callback* copia la carpeta de la corrida a Drive (`MyDrive/yolov8m_mlca/runs_agar/<corrida>`) cada `SYNC_EVERY` épocas, y de nuevo al terminar. Cada archivo se copia con un nombre temporal y se renombra, así que una desconexión a mitad de copia no deja un `last.pt` corrupto.

**Si Colab se desconecta:** ejecutá de nuevo las secciones 1 a 8 en orden (incluida la 6, que registra el MLCA). La celda:
* saltea las corridas con marca `TERMINADO`;
* si hay una corrida a medias, restaura `last.pt` desde Drive y **la retoma desde la última época copiada** (`resume=True`), con el mismo optimizador y el mismo scheduler;
* no la retoma desde cero salvo que Drive no tenga `last.pt`.

El aviso `adaptive_avg_pool2d_backward_cuda does not have a deterministic implementation` es normal: viene del pooling del MLCA y solo significa que dos corridas con la misma seed pueden no dar exactamente lo mismo.

In [ ]:
import time, glob, torch
RUNS = os.path.join(DRIVE_DIR, 'runs_agar')       # copia en Drive
LOCAL_RUNS = '/content/runs_agar'                 # donde entrena (disco local de Colab)

def sync(src, dst):
    """Copia src -> dst (solo lo que cambió). Cada archivo va a un .tmp y se renombra."""
    for root, _, files in os.walk(src):
        rel = os.path.relpath(root, src)
        out = dst if rel == '.' else os.path.join(dst, rel)
        os.makedirs(out, exist_ok=True)
        for f in files:
            s, d = os.path.join(root, f), os.path.join(out, f)
            if os.path.exists(d) and os.path.getsize(d) == os.path.getsize(s) and os.path.getmtime(d) >= os.path.getmtime(s):
                continue
            shutil.copy2(s, d + '.tmp')
            os.replace(d + '.tmp', d)

def sync_callback(local, dst):
    def cb(trainer):                              # se llama justo después de guardar last.pt / best.pt
        if (trainer.epoch + 1) % SYNC_EVERY == 0:
            try:
                sync(local, dst)
                print(f'   [Drive] época {trainer.epoch + 1} guardada en {dst}')
            except Exception as e:
                print('   [Drive] aviso: no se pudo copiar (se reintenta en la próxima época):', e)
    return cb

def tiene_optimizador(ckpt_path):
    return torch.load(ckpt_path, map_location='cpu', weights_only=False).get('optimizer') is not None

for seed in SEEDS:
    for v in VARIANTES:
        name = f'{v}_s{seed}'
        local, dst = os.path.join(LOCAL_RUNS, name), os.path.join(RUNS, name)
        best = os.path.join(dst, 'weights', 'best.pt')
        if os.path.exists(os.path.join(dst, 'TERMINADO')) and os.path.exists(best) and os.path.getsize(best) > 1e6:
            print('ya estaba:', name)
            continue
        t0 = time.time()

        # si el disco local se perdió (reinicio de Colab), restaurar la corrida desde Drive
        last_local = os.path.join(local, 'weights', 'last.pt')
        if not os.path.exists(last_local) and os.path.exists(os.path.join(dst, 'weights', 'last.pt')):
            sync(dst, local)
            print('restaurada desde Drive:', name)

        if os.path.exists(last_local) and tiene_optimizador(last_local):
            print('>>> retomando', name, 'desde la última época guardada')
            m = YOLO(last_local)
            m.add_callback('on_model_save', sync_callback(local, dst))
            m.train(resume=True)
        elif os.path.exists(last_local):
            print('>>>', name, 'ya había terminado de entrenar; solo se copia a Drive')
        else:
            print('>>> entrenando', name)
            m = make_model(v)
            m.add_callback('on_model_save', sync_callback(local, dst))
            m.train(data=TRAIN_YAML, epochs=EPOCHS, batch=BATCH, seed=seed, device=0, project=LOCAL_RUNS,
                    name=name, exist_ok=True, plots=True, **HER_ARGS)

        sync(local, dst)                                  # copia final (incluye gráficas y pesos finales)
        open(os.path.join(dst, 'TERMINADO'), 'w').write('ok')
        print(f'>>> {name} listo (esta sesión: {(time.time() - t0) / 60:.1f} min), guardado en {dst}')

## 9 · Evaluar

Para cada corrida terminada (`best.pt`, elegido por mAP50-95 en valid):

* **mAP** con `model.val` sobre **valid** (1344 fotos) y **test** (1792 fotos) de AGAR.
* **Especie por placa y conteo**: conf 0,5, NMS 0,45, especie = voto mayoritario de las cajas detectadas. **La especie verdadera sale de las etiquetas** de la placa (clase mayoritaria), porque los nombres de archivo de AGAR no la incluyen.
* `conteo_MAE`: error absoluto medio entre colonias detectadas y anotadas por foto.

Los resultados se guardan en `resultados_colab.csv` y las corridas ya evaluadas no se repiten (poné `REEVALUAR = True` para forzarlo).

In [ ]:
import glob
import numpy as np
import pandas as pd
from ultralytics import YOLO

REEVALUAR = False
CSV = os.path.join(DRIVE_DIR, 'resultados_colab.csv')

def especie_gt(lab):
    cl = [int(l.split()[0]) for l in open(lab) if len(l.split()) >= 5]
    return int(np.bincount(cl, minlength=len(NAMES)).argmax()) if cl else None, len(cl)

def placas(model, split):
    files = sorted(glob.glob(f'{ROOT}/{split}/images/*.jpg'))
    err, ok = [], []
    for i in range(0, len(files), 32):
        chunk = files[i:i + 32]
        for f, r in zip(chunk, model.predict(chunk, imgsz=640, conf=0.5, iou=0.45, batch=32, verbose=False)):
            esp, n_gt = especie_gt(f.replace('/images/', '/labels/')[:-4] + '.txt')
            cl = r.boxes.cls.cpu().numpy().astype(int)
            vote = int(np.bincount(cl, minlength=len(NAMES)).argmax()) if len(cl) else None
            err.append(len(cl) - n_gt)
            ok.append(vote is not None and vote == esp)
    return float(np.mean(ok)), float(np.mean(np.abs(err)))

prev = pd.read_csv(CSV) if (os.path.exists(CSV) and not REEVALUAR) else pd.DataFrame()
hechas = set(prev['corrida']) if len(prev) else set()
rows = prev.to_dict('records') if len(prev) else []

for d in sorted(glob.glob(os.path.join(RUNS, '*_s*'))):
    name = os.path.basename(d)
    if not os.path.exists(os.path.join(d, 'TERMINADO')) or name in hechas:
        continue
    m = YOLO(os.path.join(d, 'weights', 'best.pt'))
    row = {'corrida': name, 'variante': name.rsplit('_s', 1)[0]}
    for split, sp in (('val', 'valid'), ('test', 'test')):
        r = m.val(data=DATA_YAML, split=split, imgsz=640, batch=16, conf=0.001, iou=0.7, plots=False, verbose=False)
        row[f'{split}_mAP50'] = float(r.box.map50)
        row[f'{split}_mAP50_95'] = float(r.box.map)
        row[f'{split}_especie'], row[f'{split}_conteo_MAE'] = placas(m, sp)
    rows.append(row)
    pd.DataFrame(rows).to_csv(CSV, index=False)                 # se guarda después de cada corrida
    print(name, {k: round(v, 3) for k, v in row.items() if isinstance(v, float)})

df = pd.DataFrame(rows)
cols = ['val_mAP50', 'test_mAP50', 'test_mAP50_95', 'test_especie', 'test_conteo_MAE']
print('\nPromedio ± desvío por variante (el desvío es NaN con una sola seed):')
display(df.groupby('variante')[cols].agg(['mean', 'std']).round(3))

## 10 · Ver las gráficas del entrenamiento

Lee desde Drive, así que funciona después de una desconexión (solo hace falta ejecutar las secciones 3 y 4: configuración y Drive). También sirve para mirar una corrida **que todavía no terminó**: muestra hasta la última época copiada.

Genera `curvas_comparacion.png` en la carpeta de Drive: mAP50 y mAP50-95 por época, `base` contra `mlca`.

In [ ]:
import os, glob, shutil
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

RUNS = os.path.join(DRIVE_DIR, 'runs_agar')
CSV = os.path.join(DRIVE_DIR, 'resultados_colab.csv')

def leer(name):
    p = os.path.join(RUNS, name, 'results.csv')
    if not os.path.exists(p):
        return None
    d = pd.read_csv(p)
    d.columns = [c.strip() for c in d.columns]
    return d

nombres = sorted(os.path.basename(d) for d in glob.glob(os.path.join(RUNS, '*_s*')))
col = {'base': 'tab:blue', 'mlca': 'tab:red'}
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for n in nombres:
    d = leer(n)
    if d is None:
        continue
    v = n.rsplit('_s', 1)[0]
    ax[0].plot(d['epoch'], d['metrics/mAP50(B)'], color=col.get(v), alpha=.85, label=n)
    ax[1].plot(d['epoch'], d['metrics/mAP50-95(B)'], color=col.get(v), alpha=.85, label=n)
    ax[2].plot(d['epoch'], d['val/box_loss'] + d['val/cls_loss'] + d['val/dfl_loss'], color=col.get(v), alpha=.85, label=n)
for a, t in zip(ax, ['mAP50 (valid)', 'mAP50-95 (valid)', 'loss de validación (box+cls+dfl)']):
    a.set_title(t); a.set_xlabel('época'); a.grid(alpha=.3); a.legend()
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_DIR, 'curvas_comparacion.png'), dpi=130)
plt.show()

for n in nombres:                                            # gráficas que genera ultralytics, una corrida por vez
    for f in ('results.png', 'confusion_matrix_normalized.png'):
        p = os.path.join(RUNS, n, f)
        if os.path.exists(p):
            print(f'── {n} · {f}')
            display(Image(filename=p, width=900 if f == 'results.png' else 600))

## 11 · Bajar los resultados a tu computadora

Arma un `.zip` por corrida terminada con `best.pt`, `results.csv`, `args.yaml` y todas las gráficas (curvas, matrices de confusión, ejemplos de validación), y lo descarga. Se deja también una copia en `MyDrive/yolov8m_mlca/entregables/`.

* `last.pt` no se incluye (pesa el doble porque guarda el optimizador). Poné `INCLUIR_LAST = True` si lo querés.
* Si el navegador bloquea las descargas múltiples, permitilas para colab.research.google.com o bajá los `.zip` directo desde Drive.
* Para un solo archivo: `files.download('/content/drive/MyDrive/yolov8m_mlca/runs_agar/base_s0/weights/best.pt')`.

In [ ]:
import os, glob, shutil
from google.colab import files

RUNS = os.path.join(DRIVE_DIR, 'runs_agar')
CSV = os.path.join(DRIVE_DIR, 'resultados_colab.csv')
nombres = sorted(os.path.basename(d) for d in glob.glob(os.path.join(RUNS, '*_s*')))

INCLUIR_LAST = False
ENTREGABLES = os.path.join(DRIVE_DIR, 'entregables')
os.makedirs(ENTREGABLES, exist_ok=True)

def empaquetar(name):
    src = os.path.join(RUNS, name)
    stage = f'/content/pack/{name}'
    shutil.rmtree(stage, ignore_errors=True)
    os.makedirs(f'{stage}/weights')
    shutil.copy2(os.path.join(src, 'weights', 'best.pt'), f'{stage}/weights/best.pt')
    if INCLUIR_LAST and os.path.exists(os.path.join(src, 'weights', 'last.pt')):
        shutil.copy2(os.path.join(src, 'weights', 'last.pt'), f'{stage}/weights/last.pt')
    for f in os.listdir(src):
        if f.lower().endswith(('.png', '.jpg', '.csv', '.yaml')):
            shutil.copy2(os.path.join(src, f), stage)
    z = shutil.make_archive(os.path.join(ENTREGABLES, name), 'zip', stage)
    return z

for n in nombres:
    if os.path.exists(os.path.join(RUNS, n, 'TERMINADO')):
        z = empaquetar(n)
        print(f'{n}: {os.path.getsize(z) / 1e6:.0f} MB -> {z}')
        files.download(z)

for extra in (CSV, os.path.join(DRIVE_DIR, 'curvas_comparacion.png')):
    if os.path.exists(extra):
        files.download(extra)